# GVC Transformation Stage — D4 & D5 Domain-Conditional NLI Diagnostic

This notebook runs a diagnostic zero-shot NLI classification for **Domain 4 (Metals & Advanced Fabrication, HS chapters 72–83)** and **Domain 5 (Mechanical & Electronic, HS chapters 84–92)** only.

**Purpose:** Validate that domain-conditional hypotheses produce meaningful classification before running the full pipeline. 100 products per domain (200 total).

**Pipeline position:** Stage 0 — Zero-Shot NLI Baseline (domain-conditional).

**Model:** `cross-encoder/nli-deberta-v3-small` — open weights, runnable on Colab free tier.

**Key design decision:** Hypotheses differ by domain. Same 5 transformation stages, different hypothesis text grounded in domain-specific transformation logic. Market structure language deliberately excluded — DeBERTa infers from physical description text only.

## 0. Environment Setup (Colab)

In [1]:
# Install dependencies — run once at the top of each Colab session
!pip install -q transformers torch pandas requests scipy

In [2]:
import pandas as pd
import numpy as np
import requests
import torch
from scipy.stats import entropy as scipy_entropy

# Check GPU availability
device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if device == 0 else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: CPU


## 1. Load HS17 Product Codes

Downloads the HS 2017 product code list from BACI (CEPII). Filters to D4 (chapters 72–83) and D5 (chapters 84–92) and samples 100 products per domain.

In [3]:
# ── Download HS17 product codes from CEPII BACI ───────────────────────────────
# BACI HS17 product codes are freely available via the CEPII website.
# We use the GitHub-hosted copy for convenience in Colab.

BACI_URL = "https://raw.githubusercontent.com/datasets/baci/main/data/product_codes_HS17_V202601.csv"

try:
    df_hs17 = pd.read_csv(BACI_URL, dtype=str)
    print(f"Loaded from URL: {len(df_hs17)} rows")
except Exception as e:
    print(f"URL download failed: {e}")
    print("Falling back — please upload product_codes_HS17_V202601.csv manually.")
    from google.colab import files
    uploaded = files.upload()
    df_hs17 = pd.read_csv(list(uploaded.keys())[0], dtype=str)
    print(f"Loaded from upload: {len(df_hs17)} rows")

# Standardise column names — BACI uses 'code' and 'description'
df_hs17.columns = [c.strip().lower() for c in df_hs17.columns]
df_hs17['code'] = df_hs17['code'].str.strip().str.zfill(6)

# Keep only valid 6-digit numeric codes
df_hs17 = df_hs17[df_hs17['code'].str.match(r'^\d{6}$')].copy()
df_hs17['chapter'] = df_hs17['code'].str[:2]
df_hs17['heading'] = df_hs17['code'].str[:4]

print(f"Valid HS6 codes: {len(df_hs17):,}")
print(df_hs17.head(3))

URL download failed: HTTP Error 404: Not Found
Falling back — please upload product_codes_HS17_V202601.csv manually.


Saving product_codes_HS17_V202601.csv to product_codes_HS17_V202601.csv
Loaded from upload: 5384 rows
Valid HS6 codes: 5,384
     code                                        description chapter heading
0  010121           Horses: live, pure-bred breeding animals      01    0101
1  010129  Horses: live, other than pure-bred breeding an...      01    0101
2  010130                                        Asses: live      01    0101


## 2. Fetch HS Hierarchy (Chapter and Heading Text)

Adds chapter and heading descriptions. Uses the same GitHub CSV source as the original pipeline, with OEC API patch for any missing headings.

In [4]:
# ── Fetch chapter and heading descriptions ────────────────────────────────────
HIER_URL = "https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv"

try:
    hier = pd.read_csv(HIER_URL, dtype=str)
    hier.columns = [c.strip().lower() for c in hier.columns]
    print(f"Hierarchy CSV loaded: {len(hier)} rows, columns: {list(hier.columns)}")
except Exception as e:
    print(f"Hierarchy fetch failed: {e}")
    hier = pd.DataFrame(columns=['hscode', 'description'])

# Build chapter text lookup
# Chapters are 2-digit codes in the hierarchy file
if 'hscode' in hier.columns and 'description' in hier.columns:
    chapter_text = (
        hier[hier['hscode'].str.len() == 2]
        [['hscode', 'description']]
        .rename(columns={'hscode': 'chapter', 'description': 'chapter_text'})
        .drop_duplicates('chapter')
    )
    heading_text = (
        hier[hier['hscode'].str.len() == 4]
        [['hscode', 'description']]
        .rename(columns={'hscode': 'heading', 'description': 'heading_text'})
        .drop_duplicates('heading')
    )
else:
    # Fallback: empty lookups — description column only will be used
    chapter_text = pd.DataFrame(columns=['chapter', 'chapter_text'])
    heading_text = pd.DataFrame(columns=['heading', 'heading_text'])

print(f"Chapters with text: {len(chapter_text)}")
print(f"Headings with text: {len(heading_text)}")

Hierarchy CSV loaded: 6940 rows, columns: ['section', 'hscode', 'description', 'parent', 'level']
Chapters with text: 97
Headings with text: 1229


## 3. Layer 0 — Naive Chapter Prior

Assigns a prior transformation stage class to each product based on HS chapter membership. This is the baseline naive layer — no NLI, no text semantics. Useful for comparing against Layer 1 NLI output.

For D4 and D5, the chapter prior is fairly informative since metals chapters and machinery chapters have natural stage concentrations.

In [5]:
# ── Layer 0: Chapter-level prior transformation stage ─────────────────────────
# Assigns the modal expected transformation stage to each HS chapter.
# Based on domain structure: D4 (72-83) spans S1-S5, D5 (84-92) spans S3-S5.
# Within D4 and D5, the prior reflects where the chapter tends to cluster.

STAGE_LABELS = {
    1: "Raw/Unprocessed",
    2: "Primary Processed",
    3: "Fabricated Material",
    4: "Manufactured Component",
    5: "Assembled/Finished",
}

# Chapter prior: modal stage for each HS chapter
# D4 chapters (72-83)
# D5 chapters (84-92) — note 93+ excluded from our scope
CHAPTER_PRIOR = {
    # D4 — Metals & Advanced Fabrication
    "72": 2,  # Iron and steel — primary processed metals (pig iron, ingots, semifinished)
    "73": 3,  # Articles of iron or steel — fabricated forms (tubes, pipes, fittings)
    "74": 2,  # Copper and articles thereof — spans S2-S3
    "75": 2,  # Nickel and articles thereof
    "76": 3,  # Aluminium and articles thereof — mostly fabricated forms
    "78": 2,  # Lead and articles thereof
    "79": 2,  # Zinc and articles thereof
    "80": 2,  # Tin and articles thereof
    "81": 2,  # Other base metals
    "82": 4,  # Tools, implements — manufactured components/finished tools
    "83": 4,  # Miscellaneous articles of base metal — manufactured articles
    # D5 — Mechanical & Electronic
    "84": 4,  # Machinery and mechanical appliances — spans S4-S5
    "85": 4,  # Electrical machinery and equipment — spans S4-S5
    "86": 5,  # Railway locomotives and rolling stock — assembled
    "87": 5,  # Vehicles other than railway — assembled
    "88": 5,  # Aircraft and spacecraft — assembled
    "89": 5,  # Ships and boats — assembled
    "90": 4,  # Optical, photographic, medical instruments — components/assembled
    "91": 5,  # Clocks and watches — assembled
    "92": 5,  # Musical instruments — assembled
}

df_hs17['prior_class'] = df_hs17['chapter'].map(CHAPTER_PRIOR)
df_hs17['prior_label'] = df_hs17['prior_class'].map(STAGE_LABELS)

print("Chapter prior distribution across all HS17 chapters assigned:")
print(df_hs17.groupby(['prior_class', 'prior_label'], dropna=False)['code'].count().to_string())

Chapter prior distribution across all HS17 chapters assigned:
prior_class  prior_label           
2.0          Primary Processed          304
3.0          Fabricated Material        159
4.0          Manufactured Component    1025
5.0          Assembled/Finished         209
NaN          NaN                       3687


## 4. Filter to D4 and D5 — Sample 100 per Domain

In [6]:
# ── Domain filtering ──────────────────────────────────────────────────────────
D4_CHAPTERS = [str(i).zfill(2) for i in range(72, 84)]  # 72-83
D5_CHAPTERS = [str(i).zfill(2) for i in range(84, 93)]  # 84-92

df_d4_all = df_hs17[df_hs17['chapter'].isin(D4_CHAPTERS)].copy()
df_d5_all = df_hs17[df_hs17['chapter'].isin(D5_CHAPTERS)].copy()

print(f"D4 total products (chapters 72-83): {len(df_d4_all)}")
print(f"D5 total products (chapters 84-92): {len(df_d5_all)}")

# ── Sample 100 per domain — stratified by chapter ────────────────────────────
# Stratified sampling ensures we don't over-represent any single chapter.
# We sample proportionally, with a minimum of 2 per chapter.

SAMPLE_PER_DOMAIN = 100
RANDOM_SEED = 42

def stratified_chapter_sample(df, n, seed=42):
    """Sample n products proportionally by chapter, minimum 2 per chapter."""
    chapters = df['chapter'].value_counts()
    n_chapters = len(chapters)
    # Allocate proportionally
    alloc = (chapters / chapters.sum() * n).round().astype(int)
    # Ensure minimum of 2 per chapter
    alloc = alloc.clip(lower=2)
    # Trim to total n if over-allocated
    while alloc.sum() > n:
        alloc[alloc.idxmax()] -= 1
    sampled = []
    for ch, k in alloc.items():
        ch_df = df[df['chapter'] == ch]
        k = min(k, len(ch_df))
        sampled.append(ch_df.sample(n=k, random_state=seed))
    return pd.concat(sampled).reset_index(drop=True)

df_d4 = stratified_chapter_sample(df_d4_all, SAMPLE_PER_DOMAIN, RANDOM_SEED)
df_d5 = stratified_chapter_sample(df_d5_all, SAMPLE_PER_DOMAIN, RANDOM_SEED)

df_d4['domain'] = 'D4'
df_d5['domain'] = 'D5'

print(f"\nD4 sample: {len(df_d4)} products")
print(df_d4.groupby('chapter')['code'].count().to_string())
print(f"\nD5 sample: {len(df_d5)} products")
print(df_d5.groupby('chapter')['code'].count().to_string())

D4 total products (chapters 72-83): 563
D5 total products (chapters 84-92): 1134

D4 sample: 100 products
chapter
72    28
73    22
74     9
75     3
76     6
78     2
79     2
80     2
81     9
82    11
83     6

D5 sample: 100 products
chapter
84    44
85    23
86     2
87     8
88     2
89     2
90    13
91     4
92     2


In [7]:
# ── Merge hierarchy text and assemble diagnostic dataframe ────────────────────
df_diag = pd.concat([df_d4, df_d5], ignore_index=True)

if len(chapter_text) > 0:
    df_diag = df_diag.merge(chapter_text, on='chapter', how='left')
else:
    df_diag['chapter_text'] = None

if len(heading_text) > 0:
    df_diag = df_diag.merge(heading_text, on='heading', how='left')
else:
    df_diag['heading_text'] = None

# Rename for consistency with pipeline
df_diag = df_diag.rename(columns={'code': 'subheading', 'description': 'subheading_text'})

print(f"Total diagnostic products: {len(df_diag)}")
print(f"Missing subheading text: {df_diag['subheading_text'].isna().sum()}")
print(df_diag[['subheading', 'subheading_text', 'domain', 'prior_class', 'prior_label']].head(10))

Total diagnostic products: 200
Missing subheading text: 0
  subheading                                    subheading_text domain  \
0     721650  Iron or non-alloy steel: angles, shapes and se...     D4   
1     721730  Iron or non-alloy steel: wire, plated or coate...     D4   
2     721230  Iron or non-alloy steel: flat-rolled, width le...     D4   
3     720927  Iron or non-alloy steel: (not in coils), flat-...     D4   
4     721632  Iron or non-alloy steel: I sections, hot-rolle...     D4   
5     720529  Iron or steel, pig iron, spiegeleisen: powders...     D4   
6     721699  Iron or non-alloy steel: angles, shapes and se...     D4   
7     721310  Iron or non-alloy steel: bars and rods, hot-ro...     D4   
8     721691  Iron or non-alloy steel: angles, shapes and se...     D4   
9     721790  Iron or non-alloy steel: wire, n.e.c. in headi...     D4   

   prior_class        prior_label  
0          2.0  Primary Processed  
1          2.0  Primary Processed  
2          2.0  Pri

## 5. Domain-Conditional Hypotheses

Five transformation stages, two sets of hypotheses — one per domain. Hypotheses are grounded in physical transformation logic observable from HS6 product descriptions.

**D4 sources:** World Steel Association steelmaking process; Hulamin aluminium value chain; CMAP Metals Supply Chain Report (2018).

**D5 sources:** Sturgeon (2002) modular production networks; ISIC Rev.4 div. 26–31 explanatory notes; Athukorala (2010) parts and components.

Market structure language (commodity pricing, exchange trading) is deliberately excluded — the model infers from physical description text only.

In [8]:
# ── Domain-conditional hypothesis definitions ─────────────────────────────────

HYPOTHESES_D4 = {
    1: (
        "This product is a metallic or mineral material in its naturally occurring or extracted "
        "state, representing the primary input to industrial metal production, whose value derives "
        "entirely from its geological origin and elemental composition rather than from any "
        "industrial processing."
    ),
    2: (
        "This product is the result of a primary metallurgical transformation — such as smelting, "
        "refining, or reduction — that converts extracted ore or mineral into a standardized bulk "
        "metal whose composition and purity are defined by industrial standards."
    ),
    3: (
        "This product is a metal that has been mechanically shaped through rolling, extruding, "
        "drawing, casting, or similar forming processes into a standardized industrial form such "
        "as a sheet, plate, bar, tube, wire, or profile, suitable as an input into a wide range "
        "of downstream manufacturing processes without constituting a discrete functional part."
    ),
    4: (
        "This product is a discrete, geometrically defined metal part produced through precision "
        "manufacturing processes such as forging, stamping, machining, cutting, or welding, "
        "designed to perform a specific mechanical function and intended for incorporation into "
        "a larger assembly or system, where its value derives from dimensional precision and "
        "functional design rather than raw material content."
    ),
    5: (
        "This product is a complete metal article or machine in which multiple fabricated "
        "components have been permanently joined or assembled into a functional whole that "
        "performs an operational purpose without requiring further manufacturing transformation, "
        "and whose value derives primarily from the integration of its parts rather than from "
        "the properties of any individual material or component."
    ),
}

HYPOTHESES_D5 = {
    1: (
        "This product is a raw or unprocessed material in its natural or extracted state."
    ),
    2: (
        "This product is a material that has been purified, refined, or chemically converted "
        "into a standardised form with defined composition or properties."
    ),
    3: (
        "This product is an engineered material or structural form that has been shaped or "
        "processed into a standardised intermediate such as a substrate, film, laminate, "
        "or formed structure."
    ),
    4: (
        "This product is a discrete part or sub-assembly designed to perform a specific "
        "mechanical, electrical, or electromechanical function."
    ),
    5: (
        "This product is a complete device, machine, or system assembled from multiple "
        "components and capable of performing an operational function."
    ),
}

DOMAIN_HYPOTHESES = {
    'D4': HYPOTHESES_D4,
    'D5': HYPOTHESES_D5,
}

STAGE_LABELS = {
    1: "Raw/Unprocessed",
    2: "Primary Processed",
    3: "Fabricated Material",
    4: "Manufactured Component",
    5: "Assembled/Finished",
}

print("Hypotheses loaded.")
for domain, hyps in DOMAIN_HYPOTHESES.items():
    print(f"\n{domain}:")
    for stage, text in hyps.items():
        print(f"  S{stage} ({STAGE_LABELS[stage]}): {text[:80]}...")

Hypotheses loaded.

D4:
  S1 (Raw/Unprocessed): This product is a metallic or mineral material in its naturally occurring or ext...
  S2 (Primary Processed): This product is the result of a primary metallurgical transformation — such as s...
  S3 (Fabricated Material): This product is a metal that has been mechanically shaped through rolling, extru...
  S4 (Manufactured Component): This product is a discrete, geometrically defined metal part produced through pr...
  S5 (Assembled/Finished): This product is a complete metal article or machine in which multiple fabricated...

D5:
  S1 (Raw/Unprocessed): This product is a raw or unprocessed material in its natural or extracted state....
  S2 (Primary Processed): This product is a material that has been purified, refined, or chemically conver...
  S3 (Fabricated Material): This product is an engineered material or structural form that has been shaped o...
  S4 (Manufactured Component): This product is a discrete part or sub-assembly design

## 6. Load NLI Model

In [9]:
from transformers import pipeline

MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

print(f"Loading model: {MODEL_NAME}")
print("This may take 1-2 minutes on first run (downloading weights ~180MB)...")

nli_pipeline = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME,
    device=device,
)

print(f"Model loaded on: {'GPU' if device == 0 else 'CPU'}")

Loading model: cross-encoder/nli-deberta-v3-small
This may take 1-2 minutes on first run (downloading weights ~180MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Model loaded on: CPU


## 7. Zero-Shot NLI Classification — Domain-Conditional

Each product is classified using the hypothesis set for its domain. The model scores each hypothesis as a candidate label. We collect the full probability distribution across all 5 stages and compute:
- `nli_prob_1` … `nli_prob_5` — normalised entailment probability per stage
- `nli_class` — argmax predicted stage (1–5)
- `nli_label` — human-readable stage name
- `nli_uncertainty` — normalised entropy H / log(5), bounded [0, 1]

In [10]:
from scipy.stats import entropy as scipy_entropy
import math
from tqdm.auto import tqdm

N_STAGES = 5
LOG_N = math.log(N_STAGES)


def classify_product(description, domain):
    """
    Run zero-shot NLI for a single product description using domain-conditional hypotheses.
    Returns a dict of probabilities, predicted class, label, and uncertainty.
    """
    hypotheses = DOMAIN_HYPOTHESES[domain]
    candidate_labels = [hypotheses[s] for s in range(1, N_STAGES + 1)]

    result = nli_pipeline(
        sequences=description,
        candidate_labels=candidate_labels,
        multi_label=False,
    )

    # result['labels'] are the hypothesis texts, result['scores'] are probabilities
    # Map back to stage numbers by matching hypothesis text
    label_to_stage = {hypotheses[s]: s for s in range(1, N_STAGES + 1)}
    probs = {label_to_stage[lbl]: score
             for lbl, score in zip(result['labels'], result['scores'])}

    # Sort by stage number
    prob_vector = np.array([probs[s] for s in range(1, N_STAGES + 1)])

    # Normalise (should already sum to ~1 from pipeline but enforce)
    prob_vector = prob_vector / prob_vector.sum()

    predicted_stage = int(np.argmax(prob_vector)) + 1
    unc = scipy_entropy(prob_vector) / LOG_N  # normalised to [0,1]

    return {
        'nli_prob_1': prob_vector[0],
        'nli_prob_2': prob_vector[1],
        'nli_prob_3': prob_vector[2],
        'nli_prob_4': prob_vector[3],
        'nli_prob_5': prob_vector[4],
        'nli_class': predicted_stage,
        'nli_label': STAGE_LABELS[predicted_stage],
        'nli_uncertainty': round(unc, 4),
    }


# ── Run classification ────────────────────────────────────────────────────────
results = []

for _, row in tqdm(df_diag.iterrows(), total=len(df_diag), desc="Classifying"):
    desc = str(row['subheading_text'])
    domain = row['domain']

    if pd.isna(row['subheading_text']) or desc.strip() == '':
        results.append({
            'nli_prob_1': None, 'nli_prob_2': None, 'nli_prob_3': None,
            'nli_prob_4': None, 'nli_prob_5': None,
            'nli_class': None, 'nli_label': None, 'nli_uncertainty': None,
        })
        continue

    res = classify_product(desc, domain)
    results.append(res)

# Merge results back into diagnostic dataframe
df_results = pd.DataFrame(results)
df_layer1 = pd.concat([df_diag.reset_index(drop=True), df_results], axis=1)

print(f"\nClassification complete. {len(df_layer1)} products scored.")
print(f"Missing scores: {df_layer1['nli_class'].isna().sum()}")

Classifying:   0%|          | 0/200 [00:00<?, ?it/s]


Classification complete. 200 products scored.
Missing scores: 0


## 8. Diagnostics

### 8.1 Class Distribution

In [11]:
# ── Class distribution by domain ──────────────────────────────────────────────
print("NLI predicted stage distribution — D4:")
d4_dist = (
    df_layer1[df_layer1['domain'] == 'D4']
    .groupby(['nli_class', 'nli_label'])['subheading'].count()
    .rename('n_products')
)
print(d4_dist.to_string())

print("\nNLI predicted stage distribution — D5:")
d5_dist = (
    df_layer1[df_layer1['domain'] == 'D5']
    .groupby(['nli_class', 'nli_label'])['subheading'].count()
    .rename('n_products')
)
print(d5_dist.to_string())

# ── Agreement with chapter prior ──────────────────────────────────────────────
df_layer1['prior_nli_agree'] = (
    df_layer1['prior_class'].astype('Int64') == df_layer1['nli_class'].astype('Int64')
)
for domain in ['D4', 'D5']:
    sub = df_layer1[df_layer1['domain'] == domain]
    agree = sub['prior_nli_agree'].mean()
    print(f"\n{domain} chapter prior / NLI agreement: {agree:.1%}")
    print("(Moderate agreement expected — NLI adds within-chapter resolution)")

NLI predicted stage distribution — D4:
nli_class  nli_label             
2          Primary Processed         38
4          Manufactured Component    59
5          Assembled/Finished         3

NLI predicted stage distribution — D5:
nli_class  nli_label             
3          Fabricated Material       13
4          Manufactured Component    76
5          Assembled/Finished        11

D4 chapter prior / NLI agreement: 23.0%
(Moderate agreement expected — NLI adds within-chapter resolution)

D5 chapter prior / NLI agreement: 58.0%
(Moderate agreement expected — NLI adds within-chapter resolution)


### 8.2 Uncertainty Distribution

In [12]:
# ── Uncertainty statistics by domain ─────────────────────────────────────────
print("Uncertainty distribution by domain:")
print(
    df_layer1.groupby('domain')['nli_uncertainty']
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(3)
    .to_string()
)

# ── High uncertainty products (most informative for annotation) ───────────────
HIGH_UNC_THRESHOLD = 0.75
high_unc = df_layer1[df_layer1['nli_uncertainty'] > HIGH_UNC_THRESHOLD].copy()
print(f"\nHigh-uncertainty products (unc > {HIGH_UNC_THRESHOLD}): {len(high_unc)}")
print("These are prioritised for manual annotation in Stage 1 (active learning).")
print(
    high_unc[['subheading', 'subheading_text', 'domain', 'nli_class', 'nli_label', 'nli_uncertainty']]
    .sort_values('nli_uncertainty', ascending=False)
    .head(20)
    .to_string(index=False)
)

Uncertainty distribution by domain:
         mean  median    std    min    max
domain                                    
D4      0.643   0.632  0.095  0.465  0.908
D5      0.574   0.598  0.115  0.146  0.776

High-uncertainty products (unc > 0.75): 16
These are prioritised for manual annotation in Stage 1 (active learning).
subheading                                                                                                                                                                                                       subheading_text domain  nli_class              nli_label  nli_uncertainty
    820760                                                                                                           Tools, interchangeable: (for machine or hand tools, whether or not power-operated), for boring or broaching     D4          2      Primary Processed           0.9078
    810520                                                                                                  

### 8.3 Anchor Product Validation

Checks classification on a set of unambiguous anchor products from D4 and D5. These are products with known expected transformation stages. The model should pass on clear cases before proceeding to annotation.

In [13]:
# ── Anchor product validation ─────────────────────────────────────────────────
# HS17 codes — verified against the HS 2017 nomenclature
# D4 anchors (chapters 72-83)
# D5 anchors (chapters 84-92)

ANCHORS = {
    # D4 — S1: raw metallic ore/mineral
    "260111": (1, "D4", "Iron ores and concentrates: non-agglomerated"),
    "260600": (1, "D4", "Aluminium ores and concentrates"),
    # D4 — S2: primary processed bulk metal
    "720110": (2, "D4", "Pig iron: non-alloy, containing by weight <=0.5% phosphorus"),
    "760110": (2, "D4", "Aluminium: unwrought, not alloyed"),
    "740311": (2, "D4", "Copper: refined, cathodes and sections of cathodes"),
    # D4 — S3: mechanically shaped metal form
    "720915": (3, "D4", "Iron or steel: flat-rolled, cold-rolled, width>=600mm"),
    "760612": (3, "D4", "Aluminium: plates, sheets, strip, thickness>0.2mm, alloy"),
    "730511": (3, "D4", "Tubes, pipes: circular cross-section, iron or steel, submerged arc welded"),
    # D4 — S4: discrete precision metal part
    "848210": (4, "D4", "Ball bearings"),
    "848140": (4, "D4", "Valves: safety or relief valves"),
    "731815": (4, "D4", "Screws and bolts of iron or steel"),
    # D4 — S5: assembled metal article/machine
    "830110": (5, "D4", "Padlocks of base metal"),
    "820719": (5, "D4", "Rock drilling or earth boring tools, interchangeable"),
    # D5 — S3: engineered substrate/structural form
    "853400": (3, "D5", "Printed circuits"),
    "854390": (3, "D5", "Parts of electrical machines and apparatus"),
    # D5 — S4: discrete component/sub-assembly
    "850131": (4, "D5", "Electric motors: DC, output >37.5W but <=750W"),
    "848210": (4, "D5", "Ball bearings"),  # Also relevant in D5 context
    "854231": (4, "D5", "Processors and controllers: electronic integrated circuits"),
    "848310": (4, "D5", "Transmission shafts including cam shafts and crank shafts"),
    # D5 — S5: complete operational machine/device
    "870321": (5, "D5", "Vehicles: spark-ignition, cylinder capacity <=1000cc"),
    "851712": (5, "D5", "Telephones: smartphones"),
    "845011": (5, "D5", "Washing machines: household, capacity <=6kg"),
    "841821": (5, "D5", "Refrigerators: household, compression-type"),
    "880240": (5, "D5", "Aeroplanes and other powered aircraft, >15000kg"),
}

# Score anchors — run NLI directly on anchor descriptions
# (anchors may not be in the sample, so we score them separately)
print(f"{'Code':<8} {'Dom':<4} {'Expected':>9} {'Got':>4} {'Unc':>6}  OK?  Description")
print("-" * 110)

# First check if anchor is already in our scored sample
scored_idx = df_layer1.set_index('subheading')
anchor_results = []

for code, (expected_stage, domain, desc_hint) in ANCHORS.items():
    if code in scored_idx.index:
        row = scored_idx.loc[code]
        got = int(row['nli_class']) if pd.notna(row['nli_class']) else None
        unc = row['nli_uncertainty']
        desc_text = str(row['subheading_text'])[:55]
        source = "(sampled)"
    else:
        # Score the anchor directly using its hint description
        # Look it up in the full HS17 dataframe
        match = df_hs17[df_hs17['code'] == code]
        if len(match) > 0:
            desc_text = str(match.iloc[0]['description'])
        else:
            desc_text = desc_hint
        res = classify_product(desc_text[:200], domain)
        got = res['nli_class']
        unc = res['nli_uncertainty']
        desc_text = desc_text[:55]
        source = "(direct)"

    ok = "✓" if got == expected_stage else "✗"
    anchor_results.append({'code': code, 'expected': expected_stage, 'got': got, 'ok': ok})
    print(f"{code:<8} {domain:<4} {expected_stage:>9} {str(got):>4} {unc:>6.3f}  {ok}   {desc_text}")

n_correct = sum(1 for r in anchor_results if r['ok'] == '✓')
n_total = len(anchor_results)
print(f"\nAnchor accuracy: {n_correct}/{n_total} = {n_correct/n_total:.0%}")
print("Target: ≥70% before proceeding to annotation. ≥80% is strong.")

Code     Dom   Expected  Got    Unc  OK?  Description
--------------------------------------------------------------------------------------------------------------
260111   D4           1    4  0.723  ✗   Iron ores and concentrates: non-agglomerated
260600   D4           1    4  0.511  ✗   Aluminium ores and concentrates
720110   D4           2    4  0.577  ✗   Iron: non-alloy pig iron containing by weight 0.5% or l
760110   D4           2    4  0.850  ✗   Aluminium: unwrought, (not alloyed)
740311   D4           2    2  0.589  ✓   Copper: refined, unwrought, cathodes and sections of ca
720915   D4           3    4  0.552  ✗   Iron or non-alloy steel: in coils, flat-rolled, width 6
760612   D4           3    4  0.544  ✗   Aluminium: plates, sheets and strip, thickness exceedin
730511   D4           3    4  0.324  ✗   Iron or steel (excluding cast iron): line pipe of a kin
848210   D5           4    4  0.694  ✓   Ball bearings
848140   D4           4    5  0.726  ✗   Valves: safety or 

### 8.4 Sample Output — Inspect Classifications by Stage

In [14]:
# ── Browse predictions by stage — qualitative inspection ─────────────────────
# Shows a sample of products classified into each stage per domain.
# Use this to spot systematic errors before proceeding to annotation.

DISPLAY_COLS = ['subheading', 'subheading_text', 'nli_class', 'nli_label',
                'nli_uncertainty', 'prior_class', 'prior_label', 'prior_nli_agree']

for domain in ['D4', 'D5']:
    print(f"\n{'='*80}")
    print(f"Domain {domain} — sample classifications by stage")
    print('='*80)
    sub = df_layer1[df_layer1['domain'] == domain].copy()
    for stage in range(1, 6):
        stage_df = sub[sub['nli_class'] == stage]
        if len(stage_df) == 0:
            print(f"\nStage {stage} ({STAGE_LABELS[stage]}): No products classified here")
            continue
        print(f"\nStage {stage} ({STAGE_LABELS[stage]}): {len(stage_df)} products")
        sample = stage_df.nsmallest(5, 'nli_uncertainty') if len(stage_df) >= 5 else stage_df
        for _, r in sample.iterrows():
            flag = "" if r['prior_nli_agree'] else " [prior disagrees]"
            print(f"  {r['subheading']}  unc={r['nli_uncertainty']:.3f}{flag}")
            print(f"    {r['subheading_text']}")


Domain D4 — sample classifications by stage

Stage 1 (Raw/Unprocessed): No products classified here

Stage 2 (Primary Processed): 38 products
  830810  unc=0.493 [prior disagrees]
    Hooks, eyes and eyelets: of base metal, of a kind used for clothing, footwear, awnings, handbags, travel goods or other made up articles
  731600  unc=0.500 [prior disagrees]
    Iron or steel: anchors, grapnels and parts thereof
  732620  unc=0.538 [prior disagrees]
    Iron or steel: wire articles
  820110  unc=0.542 [prior disagrees]
    Tools, hand: spades and shovels
  720390  unc=0.547
    Ferrous products: spongy ferrous products and iron having a minimum purity by weight of 99.94%, in lumps, pellets or similar forms

Stage 3 (Fabricated Material): No products classified here

Stage 4 (Manufactured Component): 59 products
  720690  unc=0.465 [prior disagrees]
    Iron or non-alloy steel: primary forms (excluding ingots and iron of heading no. 7203)
  721012  unc=0.488 [prior disagrees]
    Iron or

### 8.5 Disagreement Analysis — Prior vs NLI

In [15]:
# ── Products where NLI disagrees with chapter prior ───────────────────────────
# These are the most interesting cases — where NLI adds within-chapter resolution.
# Inspect these to check whether NLI or the prior is more plausible.

disagree = df_layer1[~df_layer1['prior_nli_agree']].copy()
print(f"Total disagreements: {len(disagree)} / {len(df_layer1)} ({len(disagree)/len(df_layer1):.1%})")

for domain in ['D4', 'D5']:
    sub = disagree[disagree['domain'] == domain]
    print(f"\n{domain} disagreements: {len(sub)}")
    print(
        sub[['subheading', 'subheading_text', 'chapter',
             'prior_class', 'prior_label', 'nli_class', 'nli_label', 'nli_uncertainty']]
        .sort_values('nli_uncertainty', ascending=False)
        .head(15)
        .to_string(index=False)
    )

Total disagreements: 119 / 200 (59.5%)

D4 disagreements: 77
subheading                                                                                                                                                                                                       subheading_text chapter  prior_class            prior_label  nli_class              nli_label  nli_uncertainty
    820760                                                                                                           Tools, interchangeable: (for machine or hand tools, whether or not power-operated), for boring or broaching      82          4.0 Manufactured Component          2      Primary Processed           0.9078
    810520                                                                                                                        Cobalt: mattes and other intermediate products of cobalt metallurgy, unwrought cobalt, powders      81          2.0      Primary Processed          5     Assembled/Finis

## 9. Save Results

In [16]:
# ── Save diagnostic results ───────────────────────────────────────────────────
OUTPUT_FILE = "gvc_d4_d5_diagnostic_results.csv"

save_cols = [
    'subheading', 'subheading_text', 'chapter', 'heading',
    'domain', 'prior_class', 'prior_label',
    'nli_prob_1', 'nli_prob_2', 'nli_prob_3', 'nli_prob_4', 'nli_prob_5',
    'nli_class', 'nli_label', 'nli_uncertainty', 'prior_nli_agree',
]

# Add any hierarchy columns if present
for col in ['chapter_text', 'heading_text']:
    if col in df_layer1.columns:
        save_cols.insert(save_cols.index('chapter') + 1, col)
        save_cols = list(dict.fromkeys(save_cols))  # deduplicate

df_save = df_layer1[[c for c in save_cols if c in df_layer1.columns]]
df_save.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df_save)} rows to {OUTPUT_FILE}")

# Download from Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("Download triggered.")
except ImportError:
    print(f"Not in Colab — file saved locally as {OUTPUT_FILE}")

Saved 200 rows to gvc_d4_d5_diagnostic_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.


## 10. Summary and Next Steps

This cell prints a summary of the diagnostic run to inform the decision on whether to proceed to Stage 1 (annotation).

In [17]:
print("="*70)
print("DIAGNOSTIC SUMMARY")
print("="*70)

for domain in ['D4', 'D5']:
    sub = df_layer1[df_layer1['domain'] == domain]
    print(f"\nDomain {domain} ({len(sub)} products):")
    dist = sub['nli_class'].value_counts().sort_index()
    for stage, count in dist.items():
        pct = count / len(sub) * 100
        label = STAGE_LABELS.get(stage, '?')
        print(f"  S{stage} {label:<25} {count:>3} ({pct:.0f}%)")
    mean_unc = sub['nli_uncertainty'].mean()
    high_unc_n = (sub['nli_uncertainty'] > 0.75).sum()
    agree = sub['prior_nli_agree'].mean()
    print(f"  Mean uncertainty: {mean_unc:.3f}")
    print(f"  High uncertainty (>0.75): {high_unc_n} — prioritise for annotation")
    print(f"  Prior/NLI agreement: {agree:.1%}")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("""
1. Check anchor accuracy (Section 8.3):
   - ≥70%: proceed to annotation
   - <70%: review hypothesis wording before annotating

2. Inspect stage distributions (Section 8.4):
   - Severe class imbalance signals hypothesis bias
   - If one stage dominates (>50%), review that hypothesis

3. Inspect disagreements (Section 8.5):
   - NLI disagrees with chapter prior — is NLI or prior more plausible?
   - These are good candidates for annotation seed set

4. Prioritise high-uncertainty products (unc > 0.75) for manual annotation

5. If results are satisfactory:
   - Proceed to Stage 1: annotate 10-12 examples per class per domain
   - Use this diagnostic output to select boundary cases
""")

DIAGNOSTIC SUMMARY

Domain D4 (100 products):
  S2 Primary Processed          38 (38%)
  S4 Manufactured Component     59 (59%)
  S5 Assembled/Finished          3 (3%)
  Mean uncertainty: 0.643
  High uncertainty (>0.75): 14 — prioritise for annotation
  Prior/NLI agreement: 23.0%

Domain D5 (100 products):
  S3 Fabricated Material        13 (13%)
  S4 Manufactured Component     76 (76%)
  S5 Assembled/Finished         11 (11%)
  Mean uncertainty: 0.574
  High uncertainty (>0.75): 2 — prioritise for annotation
  Prior/NLI agreement: 58.0%

NEXT STEPS

1. Check anchor accuracy (Section 8.3):
   - ≥70%: proceed to annotation
   - <70%: review hypothesis wording before annotating

2. Inspect stage distributions (Section 8.4):
   - Severe class imbalance signals hypothesis bias
   - If one stage dominates (>50%), review that hypothesis

3. Inspect disagreements (Section 8.5):
   - NLI disagrees with chapter prior — is NLI or prior more plausible?
   - These are good candidates for annotati